# Demand nowcast POC — national scrapers vs JODI

Phase 3 proof of concept (**no IEA** — paywall / irregular access).

| Layer | Role |
|-------|------|
| National scrapers (DE BAFA, JP METI, KR KNOC, AU DCCEEW) | Early demand print |
| JODI `TOTDEMO` | Monthly settle / global backbone |

**Buckets:** `gasoline`, `gasoil_diesel` (same recipes as Phase 2 compare buckets).

Re-export: `python scripts/export_demand_nowcast.py`


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)


def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "analytics" / "demand_nowcast.py").exists():
            return candidate
        nested = candidate / "country_oil_scraper"
        if (nested / "analytics" / "demand_nowcast.py").exists():
            return nested
    raise RuntimeError(f"Could not locate project root from cwd: {here}")


ROOT = _resolve_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analytics.demand_nowcast import (
    build_lead_summary,
    build_nowcast_long,
    build_nowcast_tracker,
    build_nowcast_wide,
    export_nowcast_csvs,
    list_poc_series,
    make_nowcast_explorer,
)

print("ROOT:", ROOT)
display(pd.DataFrame([s.__dict__ for s in list_poc_series()]))


ROOT: c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper


,country_id,iso2,display_name,official_source,series_key,compare_bucket,jodi_energy_product,panel
0,germany,DE,Germany,BAFA,gasoline,gasoline,GASOLINE,Gasoline
1,germany,DE,Germany,BAFA,diesel,gasoil_diesel,GASDIES,Diesel
2,japan,JP,Japan,METI,gasoline,gasoline,GASOLINE,Gasoline
3,japan,JP,Japan,METI,gas_diesel,gasoil_diesel,GASDIES,Gas/diesel oil
4,korea,KR,Korea,KNOC,gasoline,gasoline,GASOLINE,Gasoline
5,korea,KR,Korea,KNOC,diesel,gasoil_diesel,GASDIES,Diesel
6,australia,AU,Australia,DCCEEW,gasoline,gasoline,GASOLINE,Gasoline
7,australia,AU,Australia,DCCEEW,diesel,gasoil_diesel,GASDIES,Diesel


## 1. Build panels + lead summary

In [2]:
paths = export_nowcast_csvs(ROOT)
long = build_nowcast_long(ROOT)
wide = build_nowcast_tracker(build_nowcast_wide(long))
lead = build_lead_summary(wide)

display(Markdown("### Export paths"))
for k, p in paths.items():
    print(f"  {k}: {p.relative_to(ROOT)}")

display(Markdown("### Lead vs JODI (months official is ahead)"))
display(lead)

display(Markdown(
    f"Long rows: **{len(long):,}** · wide: **{len(wide):,}** · "
    f"nowcast-only months: **{int(wide['is_nowcast_month'].sum()):,}**"
))


### Export paths

  wide: data\processed\demand_nowcast\demand_nowcast_poc_tableau.csv
  long: data\processed\demand_nowcast\demand_nowcast_poc_long.csv
  lead: data\processed\demand_nowcast\demand_nowcast_lead_summary.csv


### Lead vs JODI (months official is ahead)

,iso2,country_name,official_source,compare_bucket,product_label,official_last,jodi_last,lead_months,n_overlap,n_nowcast_months,nowcast_from,nowcast_to,corr,rmse,mean_bias,median_pct_diff
0,AU,Australia,DCCEEW,gasoil_diesel,Gasoil / diesel,2026-06-01,2026-06-01,0,192,0,NaT,NaT,0.981570,29.506705,19.362539,0.482937
1,AU,Australia,DCCEEW,gasoline,Gasoline,2026-06-01,2026-06-01,0,192,0,NaT,NaT,0.991823,5.029590,-3.633239,-0.979067
2,JP,Japan,METI,gasoil_diesel,Gasoil / diesel,2026-06-01,2026-06-01,0,234,0,NaT,NaT,0.999276,4.944939,-3.225322,-0.350105
3,JP,Japan,METI,gasoline,Gasoline,2026-06-01,2026-06-01,0,234,0,NaT,NaT,0.995359,10.842541,2.195443,0.167518
4,KR,Korea,KNOC,gasoil_diesel,Gasoil / diesel,2026-06-01,2026-06-01,0,294,60,1997-01-01,2001-12-01,0.992188,24.986069,-24.078674,-5.319021
5,KR,Korea,KNOC,gasoline,Gasoline,2026-06-01,2026-06-01,0,294,60,1997-01-01,2001-12-01,0.999140,1.535150,-0.492910,-0.185260
6,DE,Germany,BAFA,gasoil_diesel,Gasoil / diesel,2026-05-01,2026-06-01,-1,78,0,NaT,NaT,0.725475,282.502591,-274.828297,-28.220472
7,DE,Germany,BAFA,gasoline,Gasoline,2026-05-01,2026-06-01,-1,66,0,NaT,NaT,0.892206,64.774339,-62.427073,-13.404334


Long rows: **4,056** · wide: **2,472** · nowcast-only months: **120**

## 2. Interactive explorer

- **Levels** — official early print vs JODI settle
- **Tracker** — combined series (`official` when present, else JODI)
- **% diff** — overlap quality where both exist


In [ ]:
make_nowcast_explorer(long, wide)


Output()

## 3. How to read this

1. Positive `lead_months` → national agency is the nowcast source for the latest month(s).
2. `is_nowcast_month` in the wide CSV flags months with official data but no JODI yet.
3. When JODI arrives, compare `pct_diff` — large persistent bias means the bucket map or unit conversion needs review (Japan gasoil includes Fuel Oil A by design).
4. Inventories are **out of scope** for this POC; extend later with national CLOSTLV → JODI only (still no IEA).
